In [7]:

import pandas as pd
import torch
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import pandas_ta_classic as ta

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

df = pd.read_csv("ESFuturesAdjusted.csv")
df['time_stamp'] = pd.to_datetime(df['time_stamp'])

current_set = df.copy()

current_set['sma_20'] = current_set['close'].rolling(window=20).mean()
current_set['std'] = current_set['close'].rolling(window=20).std()
# RSI (14-period)
delta = current_set['close'].diff()
gain = delta.where(delta > 0, 0.0)
loss = -delta.where(delta < 0, 0.0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
current_set['RSI_14'] = 100 - (100 / (1 + rs))
# ATR (14-period)
high_low = current_set['high'] - current_set['low']
high_close = (current_set['high'] - current_set['close'].shift()).abs()
low_close = (current_set['low'] - current_set['close'].shift()).abs()
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
current_set['ATR_14'] = tr.rolling(window=14).mean()
current_set['typical_price'] = (current_set['high'] + current_set['low'] + current_set['close']) / 3
current_set['session_high'] = current_set.groupby(current_set['time_stamp'].dt.date)['high'].transform('max')
current_set['short_rolling'] = current_set['close'].rolling(5).mean()
current_set['long_expanding'] = current_set['close'].expanding().mean()
current_set['ema_12'] = current_set['close'].ewm(span=12, adjust=False).mean()
current_set['BB_upper'] = current_set['sma_20'] + current_set['std'] * 2
current_set['BB_lower'] = current_set['sma_20'] - current_set['std'] * 2

Using device: cpu


(5888248, 17)

In [6]:
current_set = current_set.dropna().reset_index(drop=True)
current_set = current_set.sort_values('time_stamp').reset_index(drop=True)

n = len(current_set)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = current_set.iloc[:train_end].copy()
val_df   = current_set.iloc[train_end:val_end].copy()
test_df  = current_set.iloc[val_end:].copy()

feature_cols = [
    'sma_20', 'std', 'RSI_14', 'ATR_14', 'typical_price',
    'session_high', 'short_rolling', 'long_expanding', 'ema_12',
    'BB_upper', 'BB_lower'
]
target_col = 'close'

X_train = train_df[feature_cols + ['time_stamp']]
y_train = train_df[target_col].values

X_val = val_df[feature_cols + ['time_stamp']]
y_val = val_df[target_col].values

X_test = test_df[feature_cols + ['time_stamp']]
y_test = test_df[target_col].values

class CyclicalTimeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, time_column='time_stamp'):
        self.time_column = time_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.time_column not in X_out.columns:
            return X_out

        timestamps = pd.to_datetime(X_out[self.time_column])
        minutes = timestamps.dt.minute
        hours = timestamps.dt.hour

        X_out['time_sin'] = np.sin(2 * np.pi * (hours * 60 + minutes) / (24 * 60))
        X_out['time_cos'] = np.cos(2 * np.pi * (hours * 60 + minutes) / (24 * 60))
        X_out = X_out.drop(columns=[self.time_column])
        return X_out

preprocess = ColumnTransformer(
    transformers=[
        ('time', CyclicalTimeTransformer(time_column='time_stamp'), ['time_stamp']),
        ('scaler', MinMaxScaler(), feature_cols),
    ],
    remainder='passthrough'
)

pipe = Pipeline([('preprocess', preprocess)])
pipe.fit(X_train, y_train)

X_train_processed = pipe.transform(X_train)
X_test_processed = pipe.transform(X_test)

print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")

def create_sequences(X, y, seq_length=60):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LENGTH = 60
X_train_seq, y_train_seq = create_sequences(X_train_processed, y_train, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test_processed, y_test, SEQ_LENGTH)

y_train_returns = np.diff(y_train) / y_train[:-1]
y_test_returns = np.diff(y_test) / y_test[:-1]

X_train_seq = X_train_seq[:-1]
X_test_seq = X_test_seq[:-1]

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32,device=device)
        self.y = torch.tensor(y, dtype=torch.float32,device=device).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train_seq, y_train_returns)
test_dataset = TimeSeriesDataset(X_test_seq, y_test_returns)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)


Processed train shape: (4119702, 13)
Processed test shape: (882794, 13)


KeyboardInterrupt: 

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        return self.fc(last_output)

model = LSTMModel(input_size=13, hidden_size=128, num_layers=2, dropout=0.2).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100, patience=15):
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    scaler = GradScaler()

    try:
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)

                optimizer.zero_grad()

                with autocast():
                    outputs = model(Xb)
                    loss = criterion(outputs, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for Xb, yb in val_loader:
                    Xb, yb = Xb.to(device), yb.to(device)
                    outputs = model(Xb)
                    val_loss += criterion(outputs, yb).item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)

            print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_state = model.state_dict()
                patience_counter = 0
                torch.save(model.state_dict(), "best_lstm_model.pth")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Best model saved.")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model
model = train_model(model, train_loader, test_loader, criterion, optimizer, epochs=100, patience=15)

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier()